In [3]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import glob
from scipy import linalg
from scipy.spatial.distance import cdist
import os
from PIL import Image
import math
import numpy as np

In [1]:
import os
import glob
import time
import numpy as np
import cv2

def detectTag(folder_path, marker_size):
    aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
    aruco_params = cv2.aruco.DetectorParameters()
    aruco_detector = cv2.aruco.ArucoDetector(aruco_dict, aruco_params)

    all_object_pts = []
    all_img_pts = []
    marker_size_in_world = np.array([
        [0, 0, 0],
        [marker_size, 0, 0],
        [marker_size, marker_size, 0],
        [0, marker_size, 0]
    ], dtype=np.float32)

    image_size = None
    scale = 0.1
    image_paths = glob.glob(os.path.join(folder_path, "*.jpg"))

    print(f"Found {len(image_paths)} images in {folder_path}")
    for idx, image_path in enumerate(image_paths):
        print(f"\nProcessing image {idx + 1}/{len(image_paths)}: {os.path.basename(image_path)}")
        image = cv2.imread(image_path)
        if image is None:
            print("  Failed to load image.")
            continue

        image = cv2.resize(image, (1280, 960))
        image = cv2.resize(image, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)

        if image_size is None:
            image_size = (image.shape[1], image.shape[0])  # width x height

        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        corners, ids, _ = aruco_detector.detectMarkers(image)

        if ids is not None:
            print(f"  Detected {len(ids)} marker(s): {ids.flatten().tolist()}")
            for marker_corners in corners:
                corners_2d = marker_corners.reshape(-1, 2)
                all_object_pts.append(marker_size_in_world)
                all_img_pts.append(corners_2d)
        else:
            print("  No markers detected.")

    print(f"\nTotal valid detections: {len(all_object_pts)}")
    if len(all_object_pts) == 0:
        raise ValueError("No valid marker detections. Calibration aborted.")

    print("Starting camera calibration...")
    start_time = time.time()
    error, camera_matrix, dist_coeffs, rvecs, tvecs = cv2.calibrateCamera(
        all_object_pts,
        all_img_pts,
        image_size,
        None, None
    )
    print(f"Calibration completed in {time.time() - start_time:.2f} seconds")

    return camera_matrix, dist_coeffs, error

In [ ]:
folder_path = "./0.1/converted_jpgs"
marker_size = 0.05  # in meters
camera_matrix, dist_coeffs, error = detectTag(folder_path, marker_size)
print("Camera Matrix:\n", camera_matrix)
print("Distortion Coefficients:\n", dist_coeffs)
print("Reprojection Error:", error)


Found 44 images in ./0.1/converted_jpgs

Processing image 1/44: IMG_9052.jpg
  Detected 2 marker(s): [3, 1]

Processing image 2/44: IMG_9053.jpg
  Detected 3 marker(s): [3, 1, 4]

Processing image 3/44: IMG_9054.jpg
  Detected 4 marker(s): [5, 3, 1, 0]

Processing image 4/44: IMG_9055.jpg
  Detected 6 marker(s): [5, 3, 1, 4, 2, 0]

Processing image 5/44: IMG_9056.jpg
  Detected 6 marker(s): [0, 1, 2, 3, 4, 5]

Processing image 6/44: IMG_9057.jpg
  Detected 5 marker(s): [0, 1, 2, 3, 4]

Processing image 7/44: IMG_9058.jpg
  Detected 6 marker(s): [0, 1, 2, 3, 4, 5]

Processing image 8/44: IMG_9059.jpg
  Detected 4 marker(s): [0, 1, 2, 3]

Processing image 9/44: IMG_9060.jpg
  Detected 4 marker(s): [0, 1, 2, 3]

Processing image 10/44: IMG_9061.jpg
  Detected 4 marker(s): [0, 1, 2, 3]

Processing image 11/44: IMG_9062.jpg
  Detected 4 marker(s): [5, 1, 3, 4]

Processing image 12/44: IMG_9063.jpg
  Detected 6 marker(s): [5, 1, 3, 0, 2, 4]

Processing image 13/44: IMG_9064.jpg
  Detected 4 